In [3]:
import os
from pathlib import Path

# Check Kaggle input directory for datasets
input_dir = Path('/kaggle/input')
image_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}


def count_images(root: Path, limit=None):
    """Count image files recursively under root."""
    count = 0
    for p in root.rglob('*'):
        if p.is_file() and p.suffix.lower() in image_exts:
            count += 1
            if limit is not None and count >= limit:
                return count
    return count


print('Datasets found in /kaggle/input:')
dataset_roots = []
for d in sorted(input_dir.iterdir()):
    if not d.is_dir():
        continue
    total_imgs = count_images(d)
    dataset_roots.append((d, total_imgs))
    print(f'- {d.name}: {total_imgs} images')

# Optional: set this manually if you want a specific dataset.
# Examples based on your screenshots:
#   /kaggle/input/celebahq-256-images-only
#   /kaggle/input/wikiart
#   /kaggle/input/ffhq-face-data-set
FORCE_DATASET = None

if FORCE_DATASET:
    DATA_DIR = FORCE_DATASET
else:
    non_empty = [(p, n) for p, n in dataset_roots if n > 0]
    DATA_DIR = str(max(non_empty, key=lambda x: x[1])[0]) if non_empty else None

print('Selected DATA_DIR:', DATA_DIR)
if DATA_DIR is None:
    raise ValueError('No images found in /kaggle/input. Check dataset attachment and path.')

Datasets found in /kaggle/input:
- datasets: 181446 images
Selected DATA_DIR: /kaggle/input/datasets


In [4]:
import os
import random
import torch
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader, Subset
from torch.utils.data._utils.collate import default_collate
from PIL import Image, ImageFile, UnidentifiedImageError

# Allow loading slightly damaged images instead of crashing.
ImageFile.LOAD_TRUNCATED_IMAGES = True

IMG_SIZE = 128
BATCH_SIZE = 16

# Time-budget controls (based on your measured speed).
TARGET_TOTAL_HOURS = 2.0
PLANNED_EPOCHS = 30
MEASURED_IT_PER_SEC = 1.25
SAFETY_FACTOR = 0.85  # keep margin for logging/checkpoint overhead


def is_valid_image(path):
    """Fast integrity check for image files."""
    try:
        with Image.open(path) as img:
            img.verify()
        return True
    except Exception:
        return False


class ImageDataset(Dataset):
    def __init__(self, folder, transform=None):
        if folder is None:
            raise ValueError('DATA_DIR is None. Set a valid dataset path.')

        exts = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
        raw_files = []

        # Recursive scan so nested structures work:
        # - celebahq256_imgs/train/*.jpg
        # - wikiart/wikiart/<style>/*.jpg
        # - FFHQ/thumbnails128×128/*.png
        for root, _, files in os.walk(folder):
            for f in files:
                if f.lower().endswith(exts):
                    raw_files.append(os.path.join(root, f))

        raw_files.sort()
        self.transform = transform

        # Remove corrupted files once, so workers don't crash during training.
        self.files = [p for p in raw_files if is_valid_image(p)]
        skipped = len(raw_files) - len(self.files)

        if len(self.files) == 0:
            raise ValueError(
                f'No valid image files found under {folder}. '
                'Check DATA_DIR or file integrity.'
            )

        print(f'Valid images: {len(self.files)} | Skipped corrupted: {skipped}')

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        # Return None on rare runtime decode failures; collate_fn will drop it.
        try:
            path = self.files[idx]
            img = Image.open(path).convert('RGB')
            if self.transform:
                img = self.transform(img)
            return img
        except (OSError, UnidentifiedImageError, ValueError):
            return None


def safe_collate(batch):
    batch = [x for x in batch if x is not None]
    if len(batch) == 0:
        return None
    return default_collate(batch)


transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])

full_dataset = ImageDataset(DATA_DIR, transform)

# Compute target steps/epoch and target sample count to fit 2-hour total budget.
seconds_per_epoch_budget = (TARGET_TOTAL_HOURS * 3600.0) / PLANNED_EPOCHS
TARGET_STEPS_PER_EPOCH = max(1, int(seconds_per_epoch_budget * MEASURED_IT_PER_SEC * SAFETY_FACTOR))
target_samples = TARGET_STEPS_PER_EPOCH * BATCH_SIZE

if len(full_dataset) > target_samples:
    rng = random.Random(42)
    indices = list(range(len(full_dataset)))
    rng.shuffle(indices)
    indices = indices[:target_samples]
    dataset = Subset(full_dataset, indices)
else:
    dataset = full_dataset

# Fewer workers are often more stable in Kaggle with heavy image decode.
NUM_WORKERS = 2

dataloader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    collate_fn=safe_collate,
    drop_last=True
)

print('Using reduced dataset for time budget.')
print('Planned epochs:', PLANNED_EPOCHS)
print('Target total hours:', TARGET_TOTAL_HOURS)
print('Measured it/s:', MEASURED_IT_PER_SEC)
print('Target steps/epoch:', TARGET_STEPS_PER_EPOCH)
print('Target sample count:', target_samples)
print('Actual samples used:', len(dataset))
print('Actual batches/epoch:', len(dataloader))

/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (107327830 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (99962094 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


Valid images: 181446 | Skipped corrupted: 0
Using reduced dataset for time budget.
Planned epochs: 30
Target total hours: 2.0
Measured it/s: 1.25
Target steps/epoch: 255
Target sample count: 4080
Actual samples used: 4080
Actual batches/epoch: 255


In [ ]:
import torch
import matplotlib.pyplot as plt

T = 300  # timesteps

def linear_schedule(T):
    beta_start = 1e-4
    beta_end = 0.02
    return torch.linspace(beta_start, beta_end, T)

betas = linear_schedule(T)
alphas = 1.0 - betas
alpha_cumprod = torch.cumprod(alphas, dim=0)
sqrt_alpha_cumprod = torch.sqrt(alpha_cumprod)
sqrt_one_minus_alpha_cumprod = torch.sqrt(1.0 - alpha_cumprod)

def q_sample(x0, t, noise=None):
    if noise is None:
        noise = torch.randn_like(x0)
    s1 = sqrt_alpha_cumprod[t].view(-1, 1, 1, 1).to(x0.device)
    s2 = sqrt_one_minus_alpha_cumprod[t].view(-1, 1, 1, 1).to(x0.device)
    return s1 * x0 + s2 * noise, noise

# Visualize 5 forward diffusion steps
sample_img = next(iter(dataloader))[0].unsqueeze(0)  # single image
steps = [0, 60, 120, 180, 299]

fig, axes = plt.subplots(1, len(steps), figsize=(15, 3))
for i, step in enumerate(steps):
    t = torch.tensor([step])
    noisy, _ = q_sample(sample_img, t)
    img_np = noisy[0].permute(1, 2, 0).numpy()
    img_np = (img_np * 0.5 + 0.5).clip(0, 1)
    axes[i].imshow(img_np)
    axes[i].set_title(f't={step}')
    axes[i].axis('off')
plt.suptitle('Forward Diffusion Steps')
plt.tight_layout()
plt.savefig('/kaggle/working/forward_diffusion.png', dpi=100)
plt.show()